# 01 — EDA ViANLI

Mục tiêu (khớp với Chương "Bộ dữ liệu" trong `reports/main.tex`):

1. In **schema thật** — không code tiếp khi chưa xác nhận tên cột và label mapping.
2. Phân bố nhãn theo split → quyết định có cần class weighting không.
3. Phân bố độ dài → chốt `max_length`.
4. Dò **annotation artifact**: n-gram đặc trưng theo nhãn, tỉ lệ từ phủ định.
5. Kiểm tra **data leakage** giữa train và test.
6. **Hypothesis-only baseline** — thước đo bias của dataset, và là mốc dưới cho 4 mô hình.

Mọi hình xuất ra `outputs/figures/`, số liệu xuất ra `outputs/logs/eda.json`.

## Setup Kaggle — kéo code từ GitHub

Chạy cell này **trước tiên** trên Kaggle. Bỏ qua được khi chạy ở máy local.
Sửa code ở máy → `git push` → chạy lại cell này để lấy bản mới.

In [ ]:
import os

REPO_URL = "https://github.com/dofu18/ViANLI_DL_NLP.git"

if os.path.exists("/kaggle/input"):
    !rm -rf /kaggle/working/repo
    !git clone -q $REPO_URL /kaggle/working/repo
    !cp -r /kaggle/working/repo/src /kaggle/working/
    !cp -r /kaggle/working/repo/configs /kaggle/working/
    !cp -r /kaggle/working/repo/data /kaggle/working/     # split cố định từ 01_eda
    print("src/:", sorted(os.listdir("/kaggle/working/src")))
else:
    print("Chạy local — bỏ qua bước clone.")

In [ ]:
# Kaggle: bật GPU không bắt buộc cho notebook này, nhưng cần Internet để tải dataset.
# !pip install -q underthesea datasets

import json, os, sys, collections, itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ON_KAGGLE = os.path.exists("/kaggle/input")
ROOT = "/kaggle/working" if ON_KAGGLE else os.path.abspath("..")
SRC = os.path.join(ROOT, "src")
sys.path.insert(0, SRC)

FIG_DIR = os.path.join(ROOT, "outputs", "figures")
LOG_DIR = os.path.join(ROOT, "outputs", "logs")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

from data import (LABELS, check_leakage, describe, label_to_id, load_splits,
                  normalize, set_seed, tokenize)

SEED = 42
set_seed(SEED)
eda = {"seed": SEED}
print("ROOT =", ROOT)

## 1. Schema thật

**Không bỏ qua bước này.** Nếu tên cột khác giả định, sửa `configs/*.yaml` trước khi train.

In [ ]:
ds, cols = load_splits(seed=SEED)
describe(ds)
print("\nCột đã dò được:", cols)

eda["columns"] = cols
eda["split_sizes"] = {k: len(v) for k, v in ds.items()}
eda["split_sizes"]

In [ ]:
# Đưa về DataFrame để phân tích cho nhanh
def to_df(split):
    return pd.DataFrame({
        "premise": [normalize(x) for x in split[cols["premise"]]],
        "hypothesis": [normalize(x) for x in split[cols["hypothesis"]]],
        "label": [label_to_id(y) for y in split[cols["label"]]],
    })

dfs = {name: to_df(split) for name, split in ds.items()}
for name, df in dfs.items():
    df["label_name"] = df["label"].map(lambda i: LABELS[i])

dfs["train"].head(5)

### Kiểm tra vệ sinh dữ liệu

Giá trị thiếu, câu rỗng, dòng trùng lặp hoàn toàn.

In [ ]:
sanity = {}
for name, df in dfs.items():
    sanity[name] = {
        "n": len(df),
        "premise_rỗng": int((df["premise"].str.len() == 0).sum()),
        "hypothesis_rỗng": int((df["hypothesis"].str.len() == 0).sum()),
        "dòng_trùng": int(df.duplicated(["premise", "hypothesis", "label"]).sum()),
        "cặp_trùng_khác_nhãn": int(
            df.duplicated(["premise", "hypothesis"]).sum()
            - df.duplicated(["premise", "hypothesis", "label"]).sum()
        ),
    }
eda["sanity"] = sanity
pd.DataFrame(sanity).T

## 2. Phân bố nhãn

Nếu lệch mạnh → cân nhắc `class_weights` trong config và **báo cáo macro-F1**, không chỉ accuracy.

In [ ]:
dist = pd.DataFrame({
    name: df["label_name"].value_counts(normalize=True).reindex(LABELS)
    for name, df in dfs.items()
})
display(dist.round(4))

ax = dist.plot.bar(figsize=(6, 3.5), rot=0)
ax.set_ylabel("Tỉ lệ")
ax.set_title("Phân bố nhãn theo split")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "eda_label_dist.png"), dpi=150)
plt.show()

eda["label_distribution"] = dist.round(4).to_dict()

# Mốc majority-class trên test: mọi mô hình phải vượt con số này
majority = dfs["test"]["label_name"].value_counts(normalize=True).max()
eda["majority_baseline_test"] = float(majority)
print(f"Majority baseline (test) = {majority:.4f}")

## 3. Phân bố độ dài → chốt `max_length`

In [ ]:
train = dfs["train"]
train["len_p"] = train["premise"].str.split().str.len()
train["len_h"] = train["hypothesis"].str.split().str.len()

display(train[["len_p", "len_h"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, col, title in zip(axes, ["len_p", "len_h"], ["Premise", "Hypothesis"]):
    ax.hist(train[col], bins=50)
    ax.axvline(train[col].quantile(0.95), color="red", ls="--", label="p95")
    ax.set_title(f"Độ dài {title} (số từ)")
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "eda_length_hist.png"), dpi=150)
plt.show()

p95 = {"premise": float(train["len_p"].quantile(0.95)),
       "hypothesis": float(train["len_h"].quantile(0.95))}
eda["length_p95_words"] = p95
print("Gợi ý max_length (subword ~1.5x số từ, cộng premise+hypothesis):",
      int((p95["premise"] + p95["hypothesis"]) * 1.5) + 8)

### Độ dài có tương quan với nhãn không?

Nếu có, đó cũng là một dạng artifact — mô hình có thể đoán nhãn chỉ nhờ độ dài hypothesis.

In [ ]:
by_label = train.groupby("label_name")[["len_p", "len_h"]].mean().reindex(LABELS).round(2)
display(by_label)
eda["mean_length_by_label"] = by_label.to_dict()

## 4. Annotation artifact

Giả thuyết kinh điển của NLI: hypothesis nhãn `contradiction` hay chứa từ phủ định;
nhãn `entailment` hay là bản rút gọn/khái quát của premise.

In [ ]:
NEGATION = ["không", "chẳng", "chưa", "đâu có", "không hề", "không bao giờ", "chả"]

def has_negation(text):
    low = text.lower()
    return any(w in low for w in NEGATION)

train["neg_h"] = train["hypothesis"].map(has_negation)
neg_rate = train.groupby("label_name")["neg_h"].mean().reindex(LABELS).round(4)
display(neg_rate.to_frame("tỉ lệ hypothesis có từ phủ định"))
eda["negation_rate_by_label"] = neg_rate.to_dict()

# Overlap từ vựng premise-hypothesis theo nhãn (Jaccard)
def jaccard(p, h):
    a, b = set(p.lower().split()), set(h.lower().split())
    return len(a & b) / max(len(a | b), 1)

train["overlap"] = [jaccard(p, h) for p, h in zip(train["premise"], train["hypothesis"])]
ov = train.groupby("label_name")["overlap"].mean().reindex(LABELS).round(4)
display(ov.to_frame("Jaccard overlap trung bình"))
eda["lexical_overlap_by_label"] = ov.to_dict()

In [ ]:
# Token đặc trưng nhất cho từng nhãn: PMI giữa token trong hypothesis và nhãn
counts = {lab: collections.Counter() for lab in LABELS}
for h, lab in zip(train["hypothesis"], train["label_name"]):
    counts[lab].update(set(tokenize(h, segment=False)))

total = collections.Counter()
for c in counts.values():
    total.update(c)
n_docs = len(train)

rows = []
for lab in LABELS:
    n_lab = (train["label_name"] == lab).sum()
    scored = [
        (tok, np.log((counts[lab][tok] / n_lab) / (total[tok] / n_docs)), total[tok])
        for tok in total if total[tok] >= 30 and counts[lab][tok] > 0
    ]
    top = sorted(scored, key=lambda r: -r[1])[:12]
    rows.append({"label": lab, "top_token_theo_PMI": ", ".join(t for t, _, _ in top)})

artifact_df = pd.DataFrame(rows)
display(artifact_df)
eda["top_pmi_tokens"] = artifact_df.set_index("label")["top_token_theo_PMI"].to_dict()

## 5. Data leakage

ViANLI thường tái sử dụng premise cho cả 3 nhãn → premise trùng giữa các split là bình
thường và **không** phải leakage. Cặp (premise, hypothesis) trùng nhau mới là vấn đề.

In [ ]:
leak = check_leakage(ds, cols)
print(leak)

extra = {}
for a, b in itertools.combinations(["train", "validation", "test"], 2):
    pa = set(zip(dfs[a]["premise"], dfs[a]["hypothesis"]))
    pb = set(zip(dfs[b]["premise"], dfs[b]["hypothesis"]))
    extra[f"{a}_vs_{b}_cặp_trùng"] = len(pa & pb)

eda["leakage"] = {**leak, **extra}
eda["leakage"]

## 6. Hypothesis-only baseline

TF-IDF + Logistic Regression, **chỉ nhìn hypothesis**. Nếu accuracy ≫ majority baseline
thì dataset có bias khai thác được mà không cần suy luận. Đây là con số đáng đưa vào
báo cáo và đối chiếu với 4 mô hình chính.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

def run_baseline(name, train_text, test_text):
    clf = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000),
        LogisticRegression(max_iter=1000, random_state=SEED),
    )
    clf.fit(train_text, dfs["train"]["label"])
    pred = clf.predict(test_text)
    acc = accuracy_score(dfs["test"]["label"], pred)
    f1 = f1_score(dfs["test"]["label"], pred, average="macro")
    print(f"{name:<22} acc={acc:.4f}  macro-F1={f1:.4f}")
    return {"accuracy": float(acc), "macro_f1": float(f1)}

baselines = {
    "majority": {"accuracy": float(majority), "macro_f1": None},
    "hypothesis_only": run_baseline(
        "hypothesis-only", dfs["train"]["hypothesis"], dfs["test"]["hypothesis"]),
    "premise_only": run_baseline(
        "premise-only", dfs["train"]["premise"], dfs["test"]["premise"]),
    "full_tfidf": run_baseline(
        "premise + hypothesis",
        dfs["train"]["premise"] + " [SEP] " + dfs["train"]["hypothesis"],
        dfs["test"]["premise"] + " [SEP] " + dfs["test"]["hypothesis"]),
}
eda["baselines"] = baselines
pd.DataFrame(baselines).T

## 7. Ví dụ định tính

Lấy vài cặp mỗi nhãn để đưa vào báo cáo (bảng minh họa dữ liệu).

In [ ]:
samples = (dfs["train"].groupby("label_name", group_keys=False)
           .apply(lambda g: g.sample(2, random_state=SEED))
           [["premise", "hypothesis", "label_name"]])
pd.set_option("display.max_colwidth", 120)
display(samples)

## 8. Lưu split cố định + số liệu EDA

Split được ghi ra đĩa để cả 4 mô hình dùng đúng một phân chia (nguyên tắc so sánh công bằng).

In [ ]:
SPLIT_DIR = os.path.join(ROOT, "data", "splits")
os.makedirs(SPLIT_DIR, exist_ok=True)
for name, df in dfs.items():
    path = os.path.join(SPLIT_DIR, f"{name}.csv")
    df[["premise", "hypothesis", "label"]].to_csv(path, index=False, encoding="utf-8")
    print(f"{path}  ({len(df)} dòng)")

with open(os.path.join(LOG_DIR, "eda.json"), "w", encoding="utf-8") as f:
    json.dump(eda, f, ensure_ascii=False, indent=2, default=str)
print("\nĐã ghi outputs/logs/eda.json")

## Kết luận cần điền vào báo cáo

Sau khi chạy xong, ghi lại các con số sau vào `reports/main.tex`:

| Hạng mục | Giá trị | Ảnh hưởng tới thiết kế |
|---|---|---|
| Kích thước 3 split | … | — |
| Phân bố nhãn | … | có cần `class_weights`? |
| p95 độ dài | … | chốt `max_length` trong `configs/*.yaml` |
| Tỉ lệ phủ định theo nhãn | … | bằng chứng annotation artifact |
| Cặp trùng train↔test | … | khẳng định không leakage |
| Majority baseline | … | mốc dưới tuyệt đối |
| Hypothesis-only | … | mức bias khai thác được |
| TF-IDF đầy đủ | … | mốc dưới phi-neural cho 4 mô hình |